---
## **Sujet** : Pr?diction de diagnostic de performance ?nerg?tique (DPE) d'habitats dans les d?partements de l'Ard?che (07) et de la Haute-Corse (2A)
#### **Auteurs** : M?lodie FLEURY, Mat?o PETITET 
#### **Mati?re** : UE Prog supervis?e par Vincent GUIGUE - 3A IODAA AgroParisTech
#### **Date de rendue** : 18 novembre 2024
---

##### Import des libairies n?cessaires

In [ ]:
import pandas as pd
from pandas.api.types import is_numeric_dtype
import numpy as np
import seaborn as sns
import matplotlib as plt
import time
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import *
from sklearn import svm,linear_model, naive_bayes
from sklearn.ensemble import *
import sklearn.metrics as metrics
import xgboost as xgb
import optuna

##### Les fonctions utilis?es dans ce notebook

In [ ]:
def load_file_dpe(file_dpe):
    """
    Charge un fichier csv et le transforme en DataFrame

    Input : 
    file_dpe (str) : Fichier csv ? charger

    Output :
    dpe_data (DataFrame) : Fichier transform? en DataFrame
    """
    #Permet d'afficher l'ensemble des colonnes
    pd.set_option('display.max_columns', None)
    #Lecture du jeu de donn?es (format csv)
    dpe_data = pd.read_csv(
        file_dpe,
        sep = ",")
    print("Dimension du jeu de donn?es : \n", np.shape(dpe_data)) #(49275, 131)
    #Affichage des 5 premi?res lignes du jeu de donn?es
    dpe_data.head(5)
    
    return dpe_data

def select_classe_valid(df):
    """
    S?lectionne les classes de DPE valides

    Input :
    df (DataFrame) : DataFrame avec toutes les classes de DPE

    Output :
    df (DataFrame) : DataFrame avec s?lection de classes de DPE
    """
    #Dataframe ne comportant pas les habitats ayant pour classe N
    valeur_correct = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
    df = df[(df['classe_consommation_energie'].isin(valeur_correct))].reset_index()
    
    return df

def type_conso_energie(df):
    """
    Cr?ation et remplissage de l'attribut "type_consommation_energie"

    Input :
    df (DataFrame) : DataFrame sans l'attribut "type_consommation_energie"

    Output :
    df (DataFrame) : DataFrame avec l'attribut "type_consommation_energie"
    """
    #Cr?ation attribut "type_consommation_energie"
    df["type_consommation_energie"] = ''
    #Conversion en type num?rique de l'attribut "type_consommation_energie"
    df["type_consommation_energie"] = df["type_consommation_energie"].apply(pd.to_numeric)

    #Association des classes de consommations d'?nergie aux types
    dict_classe_type = {
        'A' : 1,
        'B' : 2,
        'C' : 3,
        'D' : 4,
        'E' : 5,
        'F' : 6,
        'G' : 7
    }

    for i in range(len(df['classe_consommation_energie'])):
        df.loc[i,'type_consommation_energie'] = dict_classe_type[df.loc[i,'classe_consommation_energie']]

    return df

def supp_attrib_id(df):
    """
    Supprime les attributs de type "identifiant"

    Input :
    df (DataFrame) : DataFrame avec les attributs de type "identifiant"

    Ouput :
    df (DataFrame) : DataFrame sans les attributs de type "identifiant"
    """
    for colonne in df.columns:
        if "_id" in colonne or colonne == "id":
            print(colonne)
            df = df.drop(columns=colonne)

    return df

def supp_attrib_null(df):
    """
    Supprime les attributs ayant plus de 50% de leurs valeurs nulles

    Input :
    df (DataFrame) : DataFrame avec les attributs (valeur nulles > 50%)

    Ouput :
    df (DataFrame) : DataFrame sans les attributs (valeurs nulles > 50%)
    """
    #Fixation du seuil de suppression en nombre de lignes
    seuil_f = 0.5 * len(df)
    #Suppression des colonnes avec plus de 50% de valeurs manquantes
    colonnes_a_supprimer_f = df.columns[df.isnull().sum() > seuil_f]
    df = df.drop(columns=colonnes_a_supprimer_f)
    print(f"Colonnes supprim?es : {list(colonnes_a_supprimer_f)}")
    
    return df

def moy_val_null(df):
    """
    Supprime les valeurs nulles restantes en les rempla?ant par
    la moyenne de toutes les valeurs de l'attribut (r?serv?s
    aux attributs de type num?rique)

    Input :
    df (DataFrame) : DataFrame avec les valeurs nulles

    Output :
    df (DataFrame) : DataFrame sans valeurs nulles (seulement
    pour les attributs de type num?rique)
    """
	#Imputation des valeurs manquantes des colonnes avec valeurs restantes
    colonnes_a_imputer_f = df.columns[df.isnull().sum()>0]
    print(f"Colonnes amput?es : {list(colonnes_a_imputer_f)}")
    imputer = SimpleImputer(strategy='mean') #on utilise la moyenne
    df = pd.DataFrame(df)
    if len(colonnes_a_imputer_f) > 0 :
        for colonne in colonnes_a_imputer_f:
             if is_numeric_dtype(df[colonne]) is True: #filtre sur les attributs num?riques
                print(colonne)
                X_imput=imputer.fit_transform(df[[colonne]])
                df[[colonne]] = pd.DataFrame(X_imput, columns=[colonne], index=df.index)
                
    return df

def corr_attrib_target(df):
    """
    Fait la corr?lation entre le descripteur target et tous les autres 
    descripteurs restants. Puis, s?lectionne les descripteurs ayant 
    une valeur de corr?lation sup?rieure ? 0.1 qui seront utilis?s
    ensuite pour l'apprentissage des mod?les

    Input : 
    df (DataFrame) : DataFrame sans valeurs nulles

    Output :
    name_columns (List) : Liste de tous les descripteurs s?lectionn?s
    pour l'apprentissage
    """
    #corr?lation entre tous les attributs et la classe "type_consommation_energie"
    corr_target = df.corrwith(
        df["type_consommation_energie"], 
        method='pearson',
        drop = True)
    print(corr_target)
    corr_target_df = pd.DataFrame(corr_target).reset_index()
    corr_target_df.columns = ['attribut', 'correlation_value']
    #transformation en valeur absolue des valeurs de corr?lation
    corr_target_df["correlation_value"] = corr_target_df["correlation_value"].abs()
    #s?lection des taux de corr?lation sup?rieur ? 0.1 et tri en ordre croissant
    #Les descripteurs s?lectionn?s pour l'apprentissage
    corr_target_df[corr_target_df["correlation_value"] > 0.1].sort_values("correlation_value")
    name_columns = corr_target_df[corr_target_df["correlation_value"] > 0.1].sort_values("correlation_value").attribut.tolist()
    print("name_columns", name_columns)
    
    return name_columns

def create_df_final(df, name_columns):
    """
    Cr?ation du DataFrame final pour l'apprentissage des classifieurs

    Input : 
    df (DataFrame) : DataFrame sans valeurs nulles
    name_columns (List) : Liste de tous les descripteurs s?lectionn?s
    pour l'apprentissage

    Output : 
    dpe_df_final (DataFrame) : DataFrame final
    """
    #Cr?ation du dataframe final avec les descripteurs finaux pour cette premi?re m?thode de s?lection de variables
    dpe_df_final = pd.DataFrame()
    for name in name_columns :
        print(name)
        dpe_df_final[df.loc[:, name].name] = df.loc[:, name]
    
    return dpe_df_final


##### Chargement du fichier de donn?es (.csv) des habitats ayant fait un DPE dans le d?partement de l'Ard?che

In [ ]:
dpe_data = load_file_dpe("dep_7.csv")

<span style="color:orange"> <u> Description des donn?es : </u> Nous avons ici une base de donn?es de 49 275 habitats situ?s dans le d?partement de l'Ard?che dont on conna?t leur diagnostique de performance ?nerg?tique, et plus de 131 descripteurs dont "classe_consommation_energie" qui est l'attribut que l'on souhaite pr?dire ici. Pour plus d'informations sur la source de ces donn?es, c'est ici : [lien](https://data.ademe.fr/datasets/dpe-france). </span>

<span style="color:orange"> Quelques remarques peuvent ?tre faites sur les donn?es, il y a un grand nombre de descripteurs dont certains ont une majorit? nulle et sont d?s?quilibr?s au niveau de leur classe. Ces attributs seront soit ?limin?s soit non s?lectionn?s lors de l'?tape de r?duction de dimensionnalit?. Ensuite, il y a beaucoup d'attributs qui sont de type texte (string), ce sont donc des attributs difficiles ? manipuler car il est n?cessaire de les transformer en attribut num?rique pour le classifieur ensuite. Par exemple, le descripteur cible "classe_consommation_energie".</span>

##### Analyse et pr?-traitements du descripteur target "classe_consommation_energie"
<span style="color:orange"> Le descripteur ? pr?dire "classe_consommation_energie" prend la forme d'une lettre et va de A ? G avec A : le logement a une tr?s bonne performance ?nerg?tique et climatique, et G : le logement a une tr?s mauvaise performance ?nerg?tique et climatique. Seulement, on remarque la pr?sence de la classe "N" qui n'appartient ? aucune classe d'?nergie et qui est majoritaire dans ce jeu de donn?es (13 839 habitats). La dictionnaire des donn?es associ? ? ce jeu de donn?es [lien](https://koumoul.com/s/data-fair/api/v1/datasets/dpe-france/metadata-attachments/ADEME%20-%20DPE%20-%20Dictionnaire%20de%20donn%C3%A9es%20-%202020-06-08.pdf) ne mentionne pas cette classe. En analysant le jeu de donn?es, il semblerait que ce soit les logements ayant fait le DPE mais jug?s non admissibles. Ainsi, on va supprimer les lignes portant cette classe ou toute autre classe qui ne font pas partie d'une classe correcte (lettre de A ? G). </span>

In [ ]:
print(
    "Les diff?rentes ?tiquettes de consommation d'?nergie du jeu de donn?es sont : \n",
      dpe_data["classe_consommation_energie"].unique()
    )

print(
    "Effectif de chaque classe : \n", 
    dpe_data['classe_consommation_energie'].value_counts()
    )

print(
    "Les habitats portant la classe N : \n", 
    dpe_data[(dpe_data['classe_consommation_energie'] =='N')].head(2)
    )

dpe_data_valid = select_classe_valid(dpe_data)

print(
    "Effectif de chaque classe avec la suppression de la classe N \n", 
    dpe_data_valid['classe_consommation_energie'].value_counts()
)

<span style="color:orange"> Maintenant, ce descripteur target va ?tre transform? en un descripteur de type num?rique appel? "type_consommation_energie". Cela permet de faciliter l'apprentissage du classifieur ensuite. Concr?tement, un habitat, ayant pour classe de consommation d'?nergie la lettre A, portera le chiffre 1, un habitat, ayant pour classe de consommation d'?nergie la lettre B, portera le chiffre 2, etc.</span>

In [ ]:
dpe_data_valid = type_conso_energie(dpe_data_valid)

#### S?lection des descripteurs
<span style="color:orange"> L'?tape ici est de s?lectionner les attributs les plus pertinents pour la classe target afin d'avoir de bonnes performances avec le classifieur. En effet, selon le principe de "fl?au de dimensionnalit?", le classifieur va ?tre bon avec les donn?es d'apprentissage mais m?diocre avec les donn?es tests. </span>

##### Suppression des descripteurs de type identifiant
<span style="color:orange"> Tout d'abord, on va supprimer les attributs de type identifiant car ils n'apportent aucune information interpr?table sur le diagnostic de performance ?nerg?tique des habitats. </span>

In [ ]:
dpe_data_valid = supp_attrib_id(dpe_data_valid)

<span style="color:orange"> La liste ci-dessus montre alors les attributs type "identifiant" supprim?s. </span>

##### Gestion des valeurs manquantes
<span style="color:orange">  Dans un premier temps, on va alors s'int?resser aux valeurs nulles et supprimer les attributs qui ont plus de 50% de valeurs nulles.</span>

In [ ]:
dpe_data_valid = supp_attrib_null(dpe_data_valid)

In [ ]:
np.shape(dpe_data_valid)

<span style="color:orange"> Ce deuxi?me filtre va permettre de moyenner les valeurs manquantes des attributs num?riques seulement.</span>

In [ ]:
dpe_data_valid = moy_val_null(dpe_data_valid)

##### M?thode de corr?lation entre les descripteurs et le descripteur target
<span style="color:orange"> Cette m?thode de s?lection de variables est plut?t na?ve mais efficace. On va ?tudier la corr?lation entre les variables de la table "dpe_data_noN" et le descripteur target "type_consommation_energie". Une fois que les valeurs de corr?lation entre chaque descripteur et le descripteur target sont calcul?es, on s?lectionne les variables qui ont une valeur de corr?lation sup?rieure ? 0,1. Ces descripteurs seront utilis?s ensuite pour l'apprentissage du classifieur.</span>

In [ ]:
name_columns = corr_attrib_target(dpe_data_valid)

In [ ]:
dpe_df_final = create_df_final(df = dpe_data_valid, name_columns = name_columns)
print(dpe_df_final)

<span style="color:orange"> Finalement, avec la m?thode de corr?lation, 10 attributs sont s?lectionn?s dans l'apprentissage du classifieur. Ce sont des attributs en lien avec la consommation d'?nergie et de gaz ? effet de serre de l'habitat, avec l'ann?e de sa construction ou bien avec sa superficie et son isolation thermique. Tous ces attributs sont enregistr?s dans un dataframe final appel? : "dpe_df_final". </span>

##### S?lection des mod?les
<span style="color:orange"> Ensuite, il faut choisir le ou les mod?les qui vont pr?dire les diagnostics de performance des habitats. Ici, on va tester le mod?le de bay?sien na?f, le mod?le svm et des approches ensemblistes telles que random forest et XGBoosting. Ce sont tous des mod?les de classification supervis?e.</span>

In [ ]:
#S?paration des attributs entre ceux qui vont aider ? pr?dire (Xn) le descripteur target (y)
Xn = dpe_df_final.iloc[:, :-1]
print("Xn : \n", Xn)
y = dpe_df_final.iloc[:,-1]
print("y : \n", y)
#S?paration des donn?es d'apprentissage et des donn?es de tests
X_train, X_test, y_train, y_test = train_test_split(Xn, y, test_size=0.33, random_state=0)

In [ ]:
# Comparaison des mod?les
mod_bn = naive_bayes.GaussianNB() #mod?le bay?sien na?f
mod_svm = svm.SVC() #mod?le SVM
mod_SVM1 = svm.SVC(kernel="linear")
mod_SVM2 = svm.SVC(gamma = 10)
mod_rf = RandomForestClassifier() #mod?le random forest

# apprentissage sur les donn?es d'apprentissage
mod_bn.fit(X_train, y_train)
print("mod_bn")
mod_svm.fit(X_train, y_train)
print("mod_svm")
mod_SVM1.fit(X_train, y_train)
print("mod_SVM1")
mod_SVM2.fit(X_train, y_train)
print("mod_SVM2")
mod_rf.fit(X_train, y_train)
print(mod_rf)

In [ ]:
# pr?diction sur les donn?es tests
yhat_bn = mod_bn.predict(X_test)
yhat_svm = mod_svm.predict(X_test)
yhat1 = mod_SVM1.predict(X_test)
yhat2 = mod_SVM2.predict(X_test)
yhat_rf = mod_rf.predict(X_test)

# ?valuation de la performance des mod?les gr?ce ? la pr?cision
print("pr?cision mod?le bay?sien na?f ", metrics.accuracy_score(y_test, yhat_bn))
print("pr?cision mod?le svm ", metrics.accuracy_score(y_test, yhat_svm))
print("pr?cision mod?le svm1 ", metrics.accuracy_score(y_test, yhat1))
print("pr?cision mod?le svm2 ", metrics.accuracy_score(y_test, yhat2))
print("pr?cision mod?le random forest ", metrics.accuracy_score(y_test, yhat_rf))


In [ ]:
mod_xgb = GradientBoostingClassifier() #mod?le xgboost
mod_xgb.fit(X_train, y_train)
yhat_xgb = mod_xgb.predict(X_test)
print("pr?cision mod?le xgboost ", metrics.accuracy_score(y_test, yhat_xgb))

<span style="color:orange"> En comparant la performance des 6 classifieurs, on se rend donc compte que ce sont les mod?les svm1, random forest et xgboost qui sont les meilleurs avec 99 % de bonnes pr?dictions. </span>

##### Am?lioration de la performance
<span style="color:orange"> En compl?ment, il existe des outils int?ressants pour am?liorer la performance des classifieurs comme optuna qui selon un nombre d'it?ration choisie, s?lectionne les meilleures hyperparam?tres du mod?le. Dans notre cas, cela n'est pas n?cessaire car nous atteignons une performance proche de 1 sans am?lioration de performance. Nous avons fait le choix de laisser le code de cet outil qui est ? retrouver ci-dessous. </span>

In [ ]:
# Application d'optuna
#Define an objective function to be minimized.
def objective(trial):
    dpe_data = Xn
    dpe_target = y

    #invoke suggest methods of a trial object to generate hyperparameters (RandomForest or XGBoost)
    classifier = trial.suggest_categorical("classifier", ["RandomForest", "XGBoost"])

    #Hyperparam?tres qui vont ?tre test?s dans Optuna
    if classifier == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators", 2, 20) #2 et 20 inclus
        max_depth = int(trial.suggest_float("max_depth", 1, 32, log=True))

        clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)
    
    else:
        n_estimators = trial.suggest_int("n_estimators", 2, 20)
        max_depth = int(trial.suggest_float("max_depth", 1, 32, log=True))
        clf = GradientBoostingClassifier(n_estimators= n_estimators, max_depth= max_depth)
    
    return cross_val_score(
        clf, dpe_data, dpe_target, n_jobs=-1, cv=3
    ).mean()

#create a new study
study = optuna.create_study(direction="maximize")
#lancement de l'optimisation
study.optimize(objective, n_trials=100)
#meilleure combinaison de param?tres
trial = study.best_trial

print("Accuracy: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

Le meilleur mod?le selon optuna est le classifieur XGBoost avec 9 estimateurs et une profondeur de l'arbre ?gale ? 4.50 environ.

##### Test sur un autre d?partement : Cas de la Haute-Corse (2A)
<span style="color:orange"> Cette derni?re partie se veut pratique. On va, en effet, tester les meilleurs mod?les sur des donn?es de la m?me source mais provenant d'un autre d?partement (la Haute-Corse dans notre cas). </span>

<span style="color:orange"> Tout d'abord, certaines fonctions vont ?tre r?utilis?es pour mettre en forme l'attribut "type_consommation_energie" et supprimer les attributs et les valeurs nulles. Enfin, les m?mes descripteurs s?lectionn?s initialement pour les habitats du d?partement de l'Ard?che sont choisis pour le d?partement de la Haute-Corse. </span>

In [ ]:
# Chargement fichier des habitats du d?partement de la Haute-Corse ?valu?s pour leur performance ?nerg?tique
dpe_data_2A = load_file_dpe("dep_2A.csv")
# Selection des classes de consommation ?nerg?tique valides
dpe_data_2A_valid = select_classe_valid(dpe_data_2A)
# Cr?ation de l'attribut "type_consommation_energie"
dpe_data_2A_valid = type_conso_energie(dpe_data_2A_valid)
# Suppression des attributs ayant plus de 50% de valeurs nulles
dpe_data_2A_valid = supp_attrib_null(dpe_data_2A_valid)
# Suppression des valeurs nulles en faisant la moyenne des valeurs de l'attribut correspondant
dpe_data_2A_valid = moy_val_null(dpe_data_2A_valid)
# Creation du dataframe final en s?lectionnant les m?mes attributs que ceux du d?partement de l'Ard?che
dpe_df_final_2A = create_df_final(df = dpe_data_2A_valid, name_columns = name_columns)
print(dpe_df_final_2A)

In [ ]:
# V?rification s'il y a bien que des classes valides
dpe_data_2A_valid["classe_consommation_energie"].unique()

In [ ]:
# Tous les attributs utiles pour le classifieur
Xn_2A = dpe_df_final_2A.iloc[:, :-1]
print("Xn_2A : \n", Xn_2A)
# L'attribut objectif
y_2A = dpe_df_final_2A.iloc[:,-1]
print("y_2A : \n", y_2A)

<span style="color:orange"> S?lection des mod?les avec le maximum de pr?cision et test sur de nouveaux exemples d'apprentissages appartenant au d?partement de la Haute-Corse </span>

In [ ]:
yhat_svm1_2A = mod_SVM1.predict(Xn_2A)
yhat_rf_2A = mod_rf.predict(Xn_2A)
yhat_xgb_2A = mod_xgb.predict(Xn_2A)

print("pr?cision mod?le svm1 dep 2A ", metrics.accuracy_score(y_2A, yhat_svm1_2A))
print("pr?cision mod?le random forest dep 2A ", metrics.accuracy_score(y_2A, yhat_rf_2A))
print("pr?cision mod?le xgboost dep 2A ", metrics.accuracy_score(y_2A, yhat_xgb_2A))

<span style="color:orange"> Finalement, on se rend compte qu'apr?s avoir appliqu? les mod?les avec le plus de performance, on obtient ?galement de tr?s bon taux de pr?diction pour les habitats de la Haute-Corse. En effet, pour chacun des mod?les soit svm avec une fonction noyau lin?aire, les for?ts al?atoires et le XGBoosting, la pr?cision atteint 99 % de bonnes pr?dictions. Alors, ces mod?les pourraient s?rement ?tre appliqu?s ? l'?chelle de tous les d?partements fran?ais voire m?me permettre ? partir de quelques attributs (10 choisis ici environ) de savoir directement la valeur de diagnostic de performance ?nerg?tique de n'importe quel habitat si ces attributs sont renseign?s. </span>